# Customer Segmentation & Cohort Analysis

## Objectives
- Segment customers using K-Means clustering.
- Analyze the characteristics of each segment.
- Perform Cohort Analysis to visualize retention over time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load Data
df = pd.read_csv('../data/processed/final_analytical_dataset.csv')

## 1. Customer Segmentation (K-Means)

In [ ]:
# Select features for clustering
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Weekly_Engagement_Score', 'Avg_Monthly_Logins']
X = df[features].fillna(0)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow Method to find optimal K
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
# Apply K-Means with K=4 (Example)
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Analyze Clusters
cluster_summary = df.groupby('Cluster')[features + ['Churn']].mean()
cluster_summary

## 2. Cohort Analysis (Retention)

In [ ]:
# We need a 'CohortMonth' (Month of acquisition) and 'ActivityMonth' (Month of activity)
# Since we don't have real dates for acquisition in the public dataset (only tenure),
# we can simulate cohorts based on Tenure. 
# Tenure = 0 means joined this month. Tenure = 1 means joined last month.

# Let's assume the snapshot date is today.
snapshot_date = pd.to_datetime('today')

# Estimate Acquisition Date
df['AcquisitionDate'] = snapshot_date - pd.to_timedelta(df['tenure'] * 30, unit='D')
df['CohortMonth'] = df['AcquisitionDate'].dt.to_period('M')

# For retention, we need transaction logs, which we don't fully have.
# However, we can visualize Churn Rate by Tenure Cohort as a proxy.

cohort_data = df.groupby('Tenure_Cohort')['Churn'].mean().reset_index()
cohort_data['RetentionRate'] = 1 - cohort_data['Churn']

plt.figure(figsize=(10, 6))
sns.barplot(x='Tenure_Cohort', y='RetentionRate', data=cohort_data, palette='viridis')
plt.title('Retention Rate by Tenure Cohort')
plt.ylim(0, 1)
plt.show()